# Understanding Web Mercator Tiles

This notebook downloads map tiles from Earth Engine using the standard Web Mercator tile scheme (also called XYZ tiles). In this scheme, the world is divided into a pyramid of square tiles for each zoom level. A tile is uniquely identified by its zoom (`z`), column (`x`), and row (`y`). Because many providers adopt this same addressing system, you can fetch imagery for the same location across different services by requesting the same `z/x/y` tile coordinates. This uniformity is useful for:

- **Web mapping**: Browsers and libraries like Leaflet or MapLibre request tiles by `z/x/y`, so maps from different sources align perfectly without extra transformations.
- **Analytics and ML**: Each tile’s filename encodes its geographic footprint. You can convert a tile path (e.g., `z/x/y`) back to a bounding box or center latitude/longitude with utilities like `mercantile`, making it easy to pair pixels with real-world locations for training data.

In this notebook, we:
- Compute the set of `z/x/y` tiles that cover an Area of Interest (AOI) from GeoJSON.
- Request those tiles from Earth Engine, which serves JPEG imagery (even though the URLs omit file extensions).
- Save each tile directly as `.jpg` using the same `z/x/y` folder structure so it stays compatible with web maps and downstream ML workflows.

# Install necessary packages

These packages provide the core tools we use in this notebook:
- **requests** for fetching tiles over HTTP.
- **mercantile** to translate between geographic bounds and XYZ tile coordinates.
- **pandas/geopandas/fiona** to read CSV and GeoJSON inputs.
- **Pillow (PIL)** to inspect image responses when needed.
- **matplotlib/folium** to preview tiles and the AOI on a map.

In [ ]:
# !pip install  --upgrade tile-downloader mercantile leafmap fiona geopandas pandas pillow requests


# Imports

This code block loads the libraries used throughout the notebook:
- Data handling (`pandas`, `geopandas`)
- Paths and JSON utilities
- Mapping helpers (`folium`, `mercantile`)
- Plotting (`matplotlib`)
- HTTP requests (`requests`) and image handling (`PIL`)
- Concurrency (`ThreadPoolExecutor`) for parallel tile downloads

In [ ]:
import pandas as pd              # Tabular data (CSV of tile services)
import geopandas as gpd           # Geospatial data (GeoJSON AOI)
from pathlib import Path          # Filesystem paths (cross-platform)
import folium                     # Lightweight web map previews
import mercantile                 # XYZ tile math (bbox → tiles, tiles → bbox)
import json                       # Reading raw GeoJSON
import matplotlib.pyplot as plt   # Plotting previews of tiles
from PIL import Image             # Image inspection when previewing tiles
import requests                   # HTTP requests to fetch tiles
from io import BytesIO            # Wrap raw bytes so PIL can open them
from concurrent.futures import ThreadPoolExecutor, as_completed  # Parallel downloads

In [ ]:
# Earth Engine tile downloader - mimics browser, includes retry logic
import time

def download_earthengine_tiles(url_template: str, save_dir: Path, tiles_list, num_workers: int = 10) -> dict:
    """
    Download Earth Engine tiles and save directly as JPG.
    Earth Engine serves JPEG even without an extension in the URL.
    We mimic a browser User-Agent and retry a few times to reduce blank tiles.
    """
    headers = {
        # Pretend to be a modern browser to avoid simple blocks
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    downloaded = 0  # Counter for successful tiles
    failed = 0      # Counter for tiles that still fail after retries
    max_retries = 3

    def download_single_tile(tile):
        """Fetch one tile with retries and save as JPG."""
        nonlocal downloaded, failed
        url = url_template.replace('{z}', str(tile.z)).replace('{x}', str(tile.x)).replace('{y}', str(tile.y))

        for attempt in range(max_retries):
            try:
                time.sleep(0.1)  # Gentle pacing to avoid rate limits

                response = requests.get(url, headers=headers, timeout=30)
                response.raise_for_status()

                # Skip obviously bad/blank responses
                if len(response.content) < 100:
                    if attempt < max_retries - 1:
                        time.sleep(1)
                        continue
                    else:
                        raise Exception("Blank tile response")

                # Create z/x directory and save tile as .jpg directly (no conversion)
                tile_dir = save_dir / str(tile.z) / str(tile.x)
                tile_dir.mkdir(parents=True, exist_ok=True)
                tile_path = tile_dir / f"{tile.y}.jpg"
                tile_path.write_bytes(response.content)

                downloaded += 1
                return

            except Exception:
                if attempt == max_retries - 1:
                    failed += 1
                    print(f"    Failed tile {tile.z}/{tile.x}/{tile.y} after {max_retries} attempts")
                else:
                    time.sleep(1)  # Back off briefly before retrying

    print(f"  Downloading {len(tiles_list)} tiles with browser mimicking...")
    # Launch parallel downloads
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = [executor.submit(download_single_tile, tile) for tile in tiles_list]
        for future in as_completed(futures):
            future.result()  # Propagate any exceptions

    return {"downloaded": downloaded, "failed": failed}

# Parameters

This block collects all user-editable settings in one place so you only change values here:
- `project`: folder name under `../tilesets/` where downloads are stored.
- `tile_csv_path`: CSV listing Earth Engine tile URL templates (with `{z}`, `{x}`, `{y}` placeholders).
- `geojson_path`: GeoJSON file that defines the Area of Interest (AOI).
- `zoom_level`: tile zoom to request (higher zoom = more tiles, smaller ground footprint).
- `num_workers`: how many concurrent downloads to run.

In [ ]:
# ---- User parameters (edit these) ----
project = "mortalitree_czu_test"   # Output folder name under ../tilesets/

tile_csv_path = "../data/gee_tiles.csv"  # CSV listing Earth Engine tile URL templates
geojson_path = "../data/CZU_Fire_Perimeter_-2212360052269988636.geojson"  # AOI GeoJSON file

zoom_level = 13   # Tile zoom level (higher = more detail, more tiles)
num_workers = 10  # Parallel download threads
# --------------------------------------

# Making a list of tiles

We read the CSV of tile services and build a list of dictionaries, each holding a service name and its URL template. This keeps service info organized and easy to filter later.

In [ ]:
# Read the services CSV
df = pd.read_csv(tile_csv_path)  # Load service names and URL templates

# Build a list of dicts: [{name, url}, ...]
tile_service_urls = []
for _, row in df.iterrows():
    tile_service_urls.append({
        'name': row['name'],    # Friendly service name
        'url': row['tile_url']  # Earth Engine URL template with {z}/{x}/{y}
    })

# Quick preview to confirm we parsed correctly
print(f"Found {len(tile_service_urls)} tile services:\n")
for i, service in enumerate(tile_service_urls[:5]):  # Show first 5 entries
    print(f"{i+1}. {service['name']}")
    print(f"   URL: {service['url']}\n")

# Create an AOI

We load the GeoJSON that defines the Area of Interest, compute its bounding box (minx, miny, maxx, maxy), and show it on a quick Folium map for visual verification.

In [ ]:
# Load AOI GeoJSON from disk
with open(geojson_path, 'r') as f:
    geojson_data = json.load(f)

# Read into GeoDataFrame for convenience (enables bounds and CRS handling)
gdf = gpd.read_file(str(geojson_path))

# Bounding box: [minx, miny, maxx, maxy]
AOI = gdf.total_bounds
print(f"AOI Bounding Box: {AOI}")
print(f"West: {AOI[0]}, South: {AOI[1]}, East: {AOI[2]}, North: {AOI[3]}")

# Center point for map preview
center_lat = (AOI[1] + AOI[3]) / 2
center_lon = (AOI[0] + AOI[2]) / 2

# Create a simple Folium map to confirm the AOI
m = folium.Map(location=[center_lat, center_lon], zoom_start=10)
folium.GeoJson(geojson_data, name="AOI Features").add_to(m)

# Draw the bounding box as a rectangle
bbox_rectangle = folium.Rectangle(
    bounds=[[AOI[1], AOI[0]], [AOI[3], AOI[2]]],
    color="red",
    fill=True,
    fill_opacity=0.1,
    weight=2,
)
bbox_rectangle.add_to(m)

# Display interactive map in the notebook
m

# Create a download list

Pick which services to download. We keep it as a Python list so you can select all services or a subset without changing the rest of the notebook.

In [ ]:
# Choose which services to download
# Examples are shown below—only one option should be active at a time.

# Option 1: Download ALL services from the CSV
# download_list = tile_service_urls

# Option 2: Download specific services by index (example uses first, fifth, sixth, seventh)
# download_list = [tile_service_urls[0], tile_service_urls[4], tile_service_urls[5], tile_service_urls[6]]

# Option 2 alternative: Download specific services by index range
download_list = tile_service_urls[16:20]

# Option 3: Download services by name
# download_list = [s for s in tile_service_urls if s['name'] in ['service_a', 'service_b']]

print(f"Services selected for download: {len(download_list)}")
for service in download_list:
    print(f"  - {service['name']}")

# Download tilesets (overview)

To keep things clear for beginners, we break the download flow into small steps:
1. Prepare output folders and stop early if nothing is selected.
2. For each service, compute the `z/x/y` tiles that cover the AOI and download them in parallel.
3. Print a summary of successes and failures.
4. Preview a center tile from each service so you can visually confirm the downloads.

In [ ]:
# Step 1: Prepare output folders and guardrails
# This creates ../tilesets/<project>/ if it does not already exist and sets flags for later steps.
base_save_dir = Path("../tilesets") / project
base_save_dir.mkdir(parents=True, exist_ok=True)

# These keep state for later cells
service_status = []  # Will hold per-service results for summary and preview
skip_download = False  # When True, later cells skip work instead of erroring

# If nothing is selected, stop early so we do not run empty work in later steps
if not download_list:
    print("No services selected for download. Skipping download, summary, and preview.")
    skip_download = True
else:
    print(f"Services selected for download: {len(download_list)}")
    print(f"Tiles will be saved under: {base_save_dir.resolve()}")

## Step 2: Download tiles for each selected service

For every chosen service we:
- Build the list of `z/x/y` tiles covering the AOI at the chosen zoom.
- Download tiles in parallel (directly as `.jpg`) using the browser-mimicking downloader.
- Record successes/failures to use in the summary and preview steps.

In [ ]:
# Step 2: Download tiles in parallel for each service
# This loop downloads tiles service by service, but each service's tiles are fetched in parallel threads.
if skip_download:
    print("Skipping downloads because no services were selected.")
else:
    for service in download_list:
        print(f"\n{'='*60}")
        print(f"Downloading tiles for: {service['name']}")
        print(f"{'='*60}")

        # Prepare per-service save folder, e.g., ../tilesets/project/ServiceName
        url_template = service['url']  # Earth Engine URL template with {z}/{x}/{y}
        save_dir = base_save_dir / service['name']
        save_dir.mkdir(parents=True, exist_ok=True)

        # Compute every tile that intersects the AOI at this zoom level
        west, south, east, north = AOI
        tiles = list(mercantile.tiles(west, south, east, north, zooms=zoom_level))
        print(f"Downloading {len(tiles)} tiles at zoom level {zoom_level}...")

        try:
            # Download tiles in parallel; the helper writes each tile directly as .jpg
            result = download_earthengine_tiles(url_template, save_dir, tiles, num_workers=num_workers)
            print(f"✓ Downloaded {result['downloaded']} tiles as JPG", end="")
            if result['failed'] > 0:
                print(f" ({result['failed']} failed)")
            else:
                print()

            # Record success for later summary and preview
            service_status.append({"name": service['name'], "status": "success", "tiles": result['downloaded']})
        except Exception as exc:
            # Record failure so later steps can report it
            print(f"✗ Failed download for {service['name']}: {exc}")
            service_status.append({"name": service['name'], "status": "failed", "error": str(exc)})
            continue

    print("\nFinished running downloads for all selected services.")

## Step 3: Summarize download results

After the downloads finish, print a quick report so you can see which services succeeded or failed before previewing tiles.

In [ ]:
# Step 3: Print a summary of successes and failures
if skip_download:
    print("No services were selected, so there is nothing to summarize.")
elif not service_status:
    print("Downloads did not run or recorded no results.")
else:
    print(f"\n{'='*60}")
    print("Download summary:")
    for entry in service_status:
        if entry["status"] == "success":
            print(f"  ✓ {entry['name']} ({entry['tiles']} tiles)")
        else:
            print(f"  ✗ {entry['name']} -- {entry['status']} ({entry.get('error', 'no error reported')})")
    print(f"{'='*60}")

## Step 4: Preview a center tile from each service

Plot one tile from the center of the AOI for each service. This is a quick visual check that the downloads look right and that files were saved where expected.

In [ ]:
# Step 4: Visual preview of one tile per service (center of AOI)
if skip_download:
    print("Skipping preview because no services were selected.")
else:
    # Compute the center tile once; it is reused for all services
    center_lat = (AOI[1] + AOI[3]) / 2
    center_lon = (AOI[0] + AOI[2]) / 2
    center_tile = mercantile.tile(center_lon, center_lat, zoom_level)

    # Set up a plot column that matches the number of selected services
    num_services = len(download_list)
    if num_services == 0:
        print("No services available for preview.")
    else:
        fig, axes = plt.subplots(1, num_services, figsize=(5*num_services, 5))
        if num_services == 1:
            axes = [axes]  # Normalize to list for consistent indexing

        status_map = {entry['name']: entry for entry in service_status}

        for idx, service in enumerate(download_list):
            status = status_map.get(service['name'], {"status": "unknown"})

            # If this service did not download successfully, show a text placeholder
            if status.get("status") != "success":
                axes[idx].text(0.5, 0.5, f"No preview\n{service['name']}\n{status.get('status')}",
                               ha='center', va='center', transform=axes[idx].transAxes)
                axes[idx].axis('off')
                continue

            # Construct expected tile path: ../tilesets/project/Service/z/x/y.jpg
            tile_path = base_save_dir / service['name'] / str(center_tile.z) / str(center_tile.x) / f"{center_tile.y}.jpg"

            if tile_path.exists():
                img = Image.open(tile_path)
                axes[idx].imshow(img)
                axes[idx].set_title(service['name'], fontsize=10)
                axes[idx].axis('off')
            else:
                axes[idx].text(0.5, 0.5, f"Tile not found\n{service['name']}",
                               ha='center', va='center', transform=axes[idx].transAxes)
                axes[idx].axis('off')

        plt.tight_layout()
        plt.show()

# Print the list of downloaded services

1. print the list of 'name' and 'service_url' pairs for cut and paste by the user to view the tiles for a particular service, in the next code block 

In [ ]:
# Print list of downloaded services for reference
print("Downloaded services:\n")

# Check if download_list exists in the global scope (i.e., has been defined in a previous cell)
if "download_list" not in globals():
    print("download_list is not defined. Please run the cell that creates it (\"Create a download list from the dictionary\").")
# Check if service_status exists (this variable is created during the download process)
elif "service_status" not in globals():
    print("service_status is not defined. Please run the download cell before listing services.")
else:
    # Flag to track if we found any successfully downloaded services
    any_success = False
    
    # Loop through each service in the download list
    for service in download_list:
        # Look for this service's status in the service_status list
        # next() finds the first matching dictionary where the name matches
        status = next((s for s in service_status if s.get("name") == service["name"]), None)
        
        # Only print services that downloaded successfully
        if status and status.get("status") == "success":
            # Print the service name (formatted for easy copy-paste)
            print(f"  name: '{service['name']}'")
            # Print the URL template for this service
            print(f"  url: {service['url']}")
            # Add blank line for readability between services
            print()
            # Mark that we found at least one successful service
            any_success = True
    
    # If no services succeeded, let the user know
    if not any_success:
        print("No successfully downloaded services to list.")